# DistilBERT Classification on Gendered and Degendered Datasets

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix, roc_auc_score, average_precision_score
)
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [2]:
# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["full_text"], truncation=True, padding="max_length", max_length=512)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds)
    jaccard = jaccard_score(labels, preds, average="macro")
    hamming = hamming_loss(labels, preds)
    cm = confusion_matrix(labels, preds)

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "cohen_kappa": kappa,
        "jaccard": jaccard,
        "hamming_loss": hamming,
    }

    per_class = precision_recall_fscore_support(labels, preds, average=None)
    for i, label in enumerate(np.unique(labels)):
        metrics[f"{label}_precision"] = per_class[0][i]
        metrics[f"{label}_recall"] = per_class[1][i]
        metrics[f"{label}_f1"] = per_class[2][i]

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            metrics[f"cm_{i}{j}"] = cm[i, j]

    try:
        metrics["auc_roc"] = roc_auc_score(labels, preds, multi_class="ovr")
        metrics["auc_pr"] = average_precision_score(labels, preds)
    except:
        metrics["auc_roc"] = None
        metrics["auc_pr"] = None

    return metrics


## Gendered Dataset

In [3]:
df_gendered = pd.read_csv("data/combined_letters_gendered.csv")[["full_text", "label"]].dropna()

if df_gendered["label"].dtype == object:
    le_gendered = LabelEncoder()
    df_gendered["label"] = le_gendered.fit_transform(df_gendered["label"])


In [4]:
# Train-test split
X_train_gendered, X_test_gendered, y_train_gendered, y_test_gendered = train_test_split(
    df_gendered["full_text"],
    df_gendered["label"],
    test_size=0.2,
    stratify=df_gendered["label"],
)

In [5]:
# Dataset preparation
train_dataset_gendered = Dataset.from_dict({
    "full_text": X_train_gendered.tolist(),
    "label": y_train_gendered.tolist()
})
test_dataset_gendered = Dataset.from_dict({
    "full_text": X_test_gendered.tolist(),
    "label": y_test_gendered.tolist()
})

train_dataset_gendered = train_dataset_gendered.map(tokenize, batched=True).remove_columns("full_text")
test_dataset_gendered = test_dataset_gendered.map(tokenize, batched=True).remove_columns("full_text")


Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [6]:
# Model
model_gendered = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(set(df_gendered["label"]))
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# Training arguments
training_args_gendered = TrainingArguments(
    output_dir="./results_gendered",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to=None,
)

trainer_gendered = Trainer(
    model=model_gendered,
    args=training_args_gendered,
    train_dataset=train_dataset_gendered,
    eval_dataset=test_dataset_gendered,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3780233/911187925.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_gendered = Trainer(


In [8]:
# Training
trainer_gendered.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,0 Precision,0 Recall,0 F1,1 Precision,1 Recall,1 F1,Cm 00,Cm 01,Cm 10,Cm 11,Auc Roc,Auc Pr,Runtime,Samples Per Second,Steps Per Second
1,No log,0.005254,0.999444,0.999597,0.999102,0.999349,0.998700,0.999102,0.998699,0.998700,0.000556,1.000000,0.998205,0.999102,0.999195,1.000000,0.999597,556,1,0,1241,0.999102,0.999195,10.073600,178.487000,11.217000
2,0.030500,0.004932,0.999444,0.999597,0.999102,0.999349,0.998700,0.999102,0.998699,0.998700,0.000556,1.000000,0.998205,0.999102,0.999195,1.000000,0.999597,556,1,0,1241,0.999102,0.999195,10.066700,178.609000,11.225000
3,0.001100,0.004665,0.999444,0.999597,0.999102,0.999349,0.998700,0.999102,0.998699,0.998700,0.000556,1.000000,0.998205,0.999102,0.999195,1.000000,0.999597,556,1,0,1241,0.999102,0.999195,10.078100,178.406000,11.212000
4,0.000900,0.005206,0.999444,0.999597,0.999102,0.999349,0.998700,0.999102,0.998699,0.998700,0.000556,1.000000,0.998205,0.999102,0.999195,1.000000,0.999597,556,1,0,1241,0.999102,0.999195,10.076900,178.428000,11.214000
5,0.000400,0.005376,0.999444,0.999597,0.999102,0.999349,0.998700,0.999102,0.998699,0.998700,0.000556,1.000000,0.998205,0.999102,0.999195,1.000000,0.999597,556,1,0,1241,0.999102,0.999195,10.092900,178.145000,11.196000


TrainOutput(global_step=2250, training_loss=0.007319368705153465, metrics={'train_runtime': 624.6744, 'train_samples_per_second': 57.542, 'train_steps_per_second': 3.602, 'total_flos': 4761540644689920.0, 'train_loss': 0.007319368705153465, 'epoch': 5.0})

In [9]:
# Evaluation
trainer_gendered.evaluate(test_dataset_gendered)

{'eval_loss': 0.005375938955694437,
 'eval_accuracy': 0.9994438264738599,
 'eval_precision': 0.999597423510467,
 'eval_recall': 0.9991023339317774,
 'eval_f1': 0.9993493943903902,
 'eval_mcc': 0.9986996347258303,
 'eval_balanced_accuracy': 0.9991023339317774,
 'eval_cohen_kappa': 0.9986987892516229,
 'eval_jaccard': 0.9986997574422444,
 'eval_hamming_loss': 0.0005561735261401557,
 'eval_0_precision': 1.0,
 'eval_0_recall': 0.9982046678635548,
 'eval_0_f1': 0.9991015274034142,
 'eval_1_precision': 0.999194847020934,
 'eval_1_recall': 1.0,
 'eval_1_f1': 0.999597261377366,
 'eval_cm_00': 556,
 'eval_cm_01': 1,
 'eval_cm_10': 0,
 'eval_cm_11': 1241,
 'eval_auc_roc': 0.9991023339317774,
 'eval_auc_pr': 0.999194847020934,
 'eval_runtime': 10.087,
 'eval_samples_per_second': 178.249,
 'eval_steps_per_second': 11.202,
 'epoch': 5.0}

## Degendered Dataset

In [10]:
df_degendered = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()

if df_degendered["label"].dtype == object:
    le_degendered = LabelEncoder()
    df_degendered["label"] = le_degendered.fit_transform(df_degendered["label"])

In [11]:
# Train-test split
X_train_degendered, X_test_degendered, y_train_degendered, y_test_degendered = train_test_split(
    df_degendered["full_text"],
    df_degendered["label"],
    test_size=0.2,
    stratify=df_degendered["label"],
)


In [12]:
# Dataset preparation
train_dataset_degendered = Dataset.from_dict({
    "full_text": X_train_degendered.tolist(),
    "label": y_train_degendered.tolist()
})
test_dataset_degendered = Dataset.from_dict({
    "full_text": X_test_degendered.tolist(),
    "label": y_test_degendered.tolist()
})

train_dataset_degendered = train_dataset_degendered.map(tokenize, batched=True).remove_columns("full_text")
test_dataset_degendered = test_dataset_degendered.map(tokenize, batched=True).remove_columns("full_text")

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [13]:
# Model
model_degendered = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(set(df_degendered["label"]))
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# Training arguments
training_args_degendered = TrainingArguments(
    output_dir="./results_degendered",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to=None,
)

trainer_degendered = Trainer(
    model=model_degendered,
    args=training_args_degendered,
    train_dataset=train_dataset_degendered,
    eval_dataset=test_dataset_degendered,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3780233/3741939077.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_degendered = Trainer(


In [15]:
# Training
trainer_degendered.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,0 Precision,0 Recall,0 F1,1 Precision,1 Recall,1 F1,Cm 00,Cm 01,Cm 10,Cm 11,Auc Roc,Auc Pr,Runtime,Samples Per Second,Steps Per Second
1,No log,0.616216,0.690211,0.345106,0.500000,0.408358,0.000000,0.500000,0.000000,0.345106,0.309789,0.000000,0.000000,0.000000,0.690211,1.000000,0.816716,0,557,0,1241,0.500000,0.690211,10.071200,178.529000,11.220000
2,0.620800,0.612114,0.690211,0.345106,0.500000,0.408358,0.000000,0.500000,0.000000,0.345106,0.309789,0.000000,0.000000,0.000000,0.690211,1.000000,0.816716,0,557,0,1241,0.500000,0.690211,10.070000,178.550000,11.221000
3,0.612000,0.638599,0.690211,0.598502,0.515338,0.456530,0.077738,0.515338,0.040625,0.368749,0.309789,0.500000,0.055655,0.100162,0.697005,0.975020,0.812899,31,526,31,1210,0.515338,0.696835,10.063800,178.661000,11.228000
4,0.544400,0.718391,0.681313,0.588242,0.545009,0.531386,0.126043,0.545009,0.107775,0.407683,0.318687,0.464286,0.186715,0.266325,0.712198,0.903304,0.796448,104,453,120,1121,0.545009,0.710072,10.061500,178.701000,11.231000
5,0.437200,0.767429,0.644605,0.564234,0.555525,0.556607,0.119442,0.555525,0.117874,0.412069,0.355395,0.406818,0.321364,0.359077,0.721649,0.789686,0.754136,179,378,261,980,0.555525,0.715038,10.054300,178.830000,11.239000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

TrainOutput(global_step=2250, training_loss=0.5337065768771702, metrics={'train_runtime': 622.4448, 'train_samples_per_second': 57.748, 'train_steps_per_second': 3.615, 'total_flos': 4761540644689920.0, 'train_loss': 0.5337065768771702, 'epoch': 5.0})

In [16]:
# Evaluation
trainer_degendered.evaluate(test_dataset_degendered)


{'eval_loss': 0.7674285173416138,
 'eval_accuracy': 0.6446051167964405,
 'eval_precision': 0.5642338331771322,
 'eval_recall': 0.5555250948661603,
 'eval_f1': 0.556606718964126,
 'eval_mcc': 0.11944186336083966,
 'eval_balanced_accuracy': 0.5555250948661603,
 'eval_cohen_kappa': 0.11787360090168797,
 'eval_jaccard': 0.4120691634034109,
 'eval_hamming_loss': 0.3553948832035595,
 'eval_0_precision': 0.4068181818181818,
 'eval_0_recall': 0.3213644524236984,
 'eval_0_f1': 0.35907723169508526,
 'eval_1_precision': 0.7216494845360825,
 'eval_1_recall': 0.7896857373086221,
 'eval_1_f1': 0.7541362062331666,
 'eval_cm_00': 179,
 'eval_cm_01': 378,
 'eval_cm_10': 261,
 'eval_cm_11': 980,
 'eval_auc_roc': 0.5555250948661603,
 'eval_auc_pr': 0.715037595596844,
 'eval_runtime': 10.0322,
 'eval_samples_per_second': 179.224,
 'eval_steps_per_second': 11.264,
 'epoch': 5.0}